# Task 1.3 — Embedding & Vector Store

This notebook generates embeddings for all chunks and stores them in ChromaDB.

- **Embedding model:** BAAI/bge-small-en-v1.5 (384 dimensions)
- **Vector database:** ChromaDB with cosine similarity
- **Collections:** `recursive_chunks` (5,495) and `section_chunks` (6,118)

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q --upgrade opentelemetry-api opentelemetry-sdk
!pip install -q chromadb sentence-transformers tiktoken langchain langchain-text-splitters
print("✅ All dependencies installed")

## Cell 2 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print("✅ Drive mounted")
print(os.listdir("/content/drive/MyDrive/[3Q]Capstone1/Assignment3/"))

## Cell 3 — Load Chunks from Task 1.2

If chunk files exist locally (from running chunking.ipynb in the same session), load them directly.
Otherwise load from the saved JSON files.

In [ ]:
import json

CHUNKING_DIR = "/content/Rag-Pipeline-Over-AI-Research-Paper/part1_2_chunking"

with open(f"{CHUNKING_DIR}/chunks_recursive_clean.json") as f:
    recursive_chunks_clean = json.load(f)

with open(f"{CHUNKING_DIR}/chunks_section_clean.json") as f:
    section_chunks_clean = json.load(f)

print(f"✅ Loaded chunks")
print(f"   recursive_chunks : {len(recursive_chunks_clean):,}")
print(f"   section_chunks   : {len(section_chunks_clean):,}")

## Cell 4 — Load BGE Embedding Model

**Model:** BAAI/bge-small-en-v1.5
- 384 dimensions (lightweight, fast)
- Specifically fine-tuned for retrieval tasks (MTEB leaderboard top tier)
- Free, runs locally — no API cost
- Requires `"Represent this sentence: "` prefix for passage encoding

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Using device: {device}")

model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)

test_emb = model.encode("test sentence")
print(f"✅ Model loaded | Dimensions: {len(test_emb)}")

## Cell 5 — Setup ChromaDB

**Database:** ChromaDB with PersistentClient
- Persisted to Google Drive → survives Colab session resets
- HNSW indexing with cosine similarity
- Stores chunk text + metadata (arxiv_id, strategy, chunk_index, token_count)

In [ ]:
import chromadb

CHROMA_PATH = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/chromadb"
os.makedirs(CHROMA_PATH, exist_ok=True)

client = chromadb.PersistentClient(path=CHROMA_PATH)

# Clean up any stale collections
for col in client.list_collections():
    if col.name not in ["recursive_chunks", "section_chunks"]:
        client.delete_collection(col.name)
        print(f"🗑️  Deleted stale collection: {col.name}")

col_recursive = client.get_or_create_collection(
    name="recursive_chunks",
    metadata={"hnsw:space": "cosine"}
)
col_section = client.get_or_create_collection(
    name="section_chunks",
    metadata={"hnsw:space": "cosine"}
)

print(f"✅ ChromaDB ready at: {CHROMA_PATH}")
print(f"   recursive_chunks : {col_recursive.count()} docs")
print(f"   section_chunks   : {col_section.count()} docs")

## Cell 6 — Embed & Store All Chunks

In [ ]:
from tqdm import tqdm

def embed_and_store(chunks: list, collection, batch_size: int = 64):
    """Embed chunks in batches and store in ChromaDB with metadata.
    Skips already-stored chunks so it's safe to re-run.
    """
    existing_ids = set(collection.get()["ids"])
    new_chunks   = [c for c in chunks if c["chunk_id"] not in existing_ids]

    if not new_chunks:
        print(f"✅ Already stored all chunks in '{collection.name}'")
        return

    print(f"📥 Embedding {len(new_chunks):,} chunks → '{collection.name}'...")

    for i in tqdm(range(0, len(new_chunks), batch_size)):
        batch     = new_chunks[i : i + batch_size]
        texts     = [c["text"] for c in batch]
        ids       = [c["chunk_id"] for c in batch]
        metadatas = [{
            "arxiv_id"    : c["arxiv_id"],
            "strategy"    : c["strategy"],
            "chunk_index" : c["chunk_index"],
            "token_count" : c["token_count"]
        } for c in batch]

        # BGE requires 'Represent this sentence: ' prefix for passage encoding
        embeddings = model.encode(
            [f"Represent this sentence: {t}" for t in texts],
            normalize_embeddings=True,
            show_progress_bar=False
        ).tolist()

        collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=texts,
            metadatas=metadatas
        )

    print(f"✅ Done! Total stored: {collection.count():,}")


embed_and_store(recursive_chunks_clean, col_recursive)
embed_and_store(section_chunks_clean,   col_section)

## Cell 7 — Verify Storage

In [ ]:
print(f"recursive_chunks : {col_recursive.count():,}")
print(f"section_chunks   : {col_section.count():,}")

db_file = f"{CHROMA_PATH}/chroma.sqlite3"
size_mb = os.path.getsize(db_file) / (1024*1024)
print(f"\nchroma.sqlite3 size: {size_mb:.1f} MB")

if size_mb > 10:
    print("✅ Successfully saved to Drive!")
else:
    print("❌ Not saved properly — re-run Cell 6")

## Cell 8 — Task 1.2 Retrieval Performance Comparison

In [ ]:
import numpy as np

eval_queries = [
    "How does self-attention mechanism work in transformers?",
    "What methods are used for graph neural network node classification?",
    "How is RLHF used to align large language models with human feedback?",
    "What are the trade-offs between sparse and dense retrieval methods?",
    "How do diffusion models generate images from noise?"
]
TOP_K = 5

def evaluate_strategy(collection, queries, k=5):
    all_scores, all_token_lens, all_unique_docs = [], [], []
    for query in queries:
        query_emb = model.encode(query, normalize_embeddings=True).tolist()
        results   = collection.query(
            query_embeddings=[query_emb],
            n_results=k,
            include=["documents", "metadatas", "distances"]
        )
        similarities  = [1 - d for d in results["distances"][0]]
        token_lens    = [m["token_count"] for m in results["metadatas"][0]]
        unique_papers = len(set(m["arxiv_id"] for m in results["metadatas"][0]))
        all_scores.append(np.mean(similarities))
        all_token_lens.append(np.mean(token_lens))
        all_unique_docs.append(unique_papers)
    return {
        "mean_similarity"     : np.mean(all_scores),
        "mean_chunk_tokens"   : np.mean(all_token_lens),
        "mean_unique_papers"  : np.mean(all_unique_docs),
        "per_query_similarity": all_scores
    }

print("⏳ Evaluating retrieval performance...")
rec_eval = evaluate_strategy(col_recursive, eval_queries, TOP_K)
sec_eval = evaluate_strategy(col_section,   eval_queries, TOP_K)

print(f"\n{'='*62}")
print(f"{'Metric':<30} {'Recursive':>14} {'Section-Aware':>14}")
print(f"{'='*62}")
print(f"{'Mean similarity score':<30} {rec_eval['mean_similarity']:>14.4f} {sec_eval['mean_similarity']:>14.4f}")
print(f"{'Mean chunk tokens':<30} {rec_eval['mean_chunk_tokens']:>14.1f} {sec_eval['mean_chunk_tokens']:>14.1f}")
print(f"{'Mean unique papers hit':<30} {rec_eval['mean_unique_papers']:>14.1f} {sec_eval['mean_unique_papers']:>14.1f}")
print(f"{'='*62}")

print(f"\n📊 Per-Query Scores:")
print(f"  {'Query':<52} {'Recur':>6} {'Sect':>6}")
print(f"  {'-'*52} {'-'*6} {'-'*6}")
for i, q in enumerate(eval_queries):
    r = rec_eval["per_query_similarity"][i]
    s = sec_eval["per_query_similarity"][i]
    winner = "◀ R" if r > s else "◀ S"
    print(f"  {q[:52]:<52} {r:>6.4f} {s:>6.4f}  {winner}")

## Cell 9 — Task 1.3 Verify: 3 Benchmark Queries with Hit@5

In [ ]:
import json, random

BENCHMARK_PATH = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3"

with open(f"{BENCHMARK_PATH}/queries.json") as f:
    queries = json.load(f)
with open(f"{BENCHMARK_PATH}/qrels.json") as f:
    qrels = json.load(f)
with open(f"{BENCHMARK_PATH}/answers.json") as f:
    answers = json.load(f)

matched_uuids = [uid for uid in queries if uid in qrels]
random.seed(42)
sample_uuids  = random.sample(matched_uuids, 3)

def verify_query(uuid, collection, k=5):
    query_text = queries[uuid]["query"]
    gt_doc_id  = qrels[uuid]["doc_id"]
    gt_answer  = answers.get(uuid, "N/A")

    query_emb = model.encode(query_text, normalize_embeddings=True).tolist()
    results   = collection.query(
        query_embeddings=[query_emb],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    retrieved_docs = [m["arxiv_id"] for m in results["metadatas"][0]]
    hit            = gt_doc_id in retrieved_docs
    hit_rank       = retrieved_docs.index(gt_doc_id) + 1 if hit else None
    similarities   = [round(1 - d, 4) for d in results["distances"][0]]

    print(f"\n{'='*65}")
    print(f"🔍 Query  : {query_text}")
    print(f"   GT Doc : {gt_doc_id}")
    print(f"   Answer : {str(gt_answer)[:100]}...")
    print(f"{'='*65}")
    print(f"   Hit@{k} : {'✅ YES (rank #'+str(hit_rank)+')' if hit else '❌ NO'}")
    print(f"\n   Top {k} Retrieved Chunks:")
    for i, (doc, meta, sim) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        similarities
    )):
        marker = " ← ✅ CORRECT" if meta["arxiv_id"] == gt_doc_id else ""
        print(f"\n   Rank #{i+1} | {meta['arxiv_id']} | sim={sim}{marker}")
        print(f"   {doc[:150]}...")
    return hit

for strategy, collection in [
    ("RECURSIVE",     col_recursive),
    ("SECTION-AWARE", col_section)
]:
    print(f"\n\n{'#'*65}")
    print(f"  STRATEGY: {strategy}")
    print(f"{'#'*65}")
    hits = sum(int(verify_query(uid, collection, TOP_K)) for uid in sample_uuids)
    print(f"\n📊 {strategy} — Hit@{TOP_K}: {hits}/3 correct docs retrieved")

## Cell 10 — Save Embedding Config

In [ ]:
import json

EMBEDDING_DIR = "/content/Rag-Pipeline-Over-AI-Research-Paper/part1_3_embedding"
os.makedirs(EMBEDDING_DIR, exist_ok=True)

embedding_config = {
    "model"       : "BAAI/bge-small-en-v1.5",
    "dimensions"  : 384,
    "vector_db"   : "ChromaDB",
    "similarity"  : "cosine",
    "chroma_path" : CHROMA_PATH,
    "collections" : {
        "recursive_chunks": col_recursive.count(),
        "section_chunks"  : col_section.count()
    }
}

with open(f"{EMBEDDING_DIR}/embedding_config.json", "w") as f:
    json.dump(embedding_config, f, indent=2)

print("✅ Embedding config saved")
print(json.dumps(embedding_config, indent=2))